In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import cv2

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


In [3]:
# MediaPipe hand connections 
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),        # thumb
    (0,5),(5,6),(6,7),(7,8),        # index
    (0,9),(9,10),(10,11),(11,12),   # middle  
    (0,13),(13,14),(14,15),(15,16), # ring    
    (0,17),(17,18),(18,19),(19,20), # pinky
    (5,9),(9,13),(13,17)            # palm
]

In [4]:
print("numpy:", np.__version__)
print("opencv:", cv2.__version__)
print("mediapipe:", mp.__version__)

numpy: 2.4.2
opencv: 4.13.0
mediapipe: 0.10.32


#### Mediapipe Hand Landmarks in Real Time

In [6]:
model_path = os.path.join("mp", "hand_landmarker.task")
if not os.path.isfile(model_path):
    raise FileNotFoundError(
        f"Could not find {model_path} in {os.getcwd()}. Put the model next to this notebook or use an absolute path."
    )

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    running_mode=vision.RunningMode.IMAGE,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    )

detector = vision.HandLandmarker.create_from_options(options)


cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Check camera permissions and close other apps using the camera.")

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    h, w, _ = frame.shape
    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            points = []
            for lm in hand_landmarks:
                x = int(lm.x * w)
                y = int(lm.y * h)
                points.append((x, y))
                cv2.circle(frame, (x, y), 4, (0, 255, 0), -1)

            for start_idx, end_idx in HAND_CONNECTIONS:
                x1, y1 = points[start_idx]
                x2, y2 = points[end_idx]
                cv2.line(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

    cv2.putText(frame, "Press q to quit", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.imshow("MediaPipe Hand Landmarks", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()